# IntelliML Platform - Model Comparison Notebook
This notebook uses RAW data - no cleaning, exactly as the platform would process it.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Try importing XGBoost, LightGBM, CatBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except:
    LGB_AVAILABLE = False

try:
    from catboost import CatBoostClassifier
    CAT_AVAILABLE = True
except:
    CAT_AVAILABLE = False

print(f"XGBoost: {XGB_AVAILABLE}, LightGBM: {LGB_AVAILABLE}, CatBoost: {CAT_AVAILABLE}")

## Load Data (TRULY RAW - No Cleaning)

In [ ]:
# Load data - RAW, no cleaning at all
df = pd.read_csv('data/titanic.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

In [ ]:
# RAW preprocessing - EXACTLY like the platform does it
df_raw = df.copy()

# Target
y = df_raw['Survived']
X = df_raw.drop('Survived', axis=1)

# Platform logic (model_trainer.py lines 544-561):
# 1. Get categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# 2. Drop high-cardinality columns (>10 unique values) FIRST
cols_to_drop = []
for col in categorical_cols:
    if X[col].nunique() > 10:
        cols_to_drop.append(col)

X = X.drop(columns=cols_to_drop)
print(f"Dropped columns: {cols_to_drop}")

# 3. Then apply get_dummies + fillna
X_processed = pd.get_dummies(X, drop_first=True).fillna(0)

print(f"Processed features: {X_processed.shape}")
print(f"Feature columns: {list(X_processed.columns)}")
X_processed.head()

In [ ]:
# Train-test split (same as platform)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Train All Algorithms

In [ ]:
# Define algorithms (matching platform)
algorithms = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
}

if XGB_AVAILABLE:
    algorithms['XGBoost'] = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)

if LGB_AVAILABLE:
    algorithms['LightGBM'] = lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)

if CAT_AVAILABLE:
    algorithms['CatBoost'] = CatBoostClassifier(n_estimators=100, random_state=42, verbose=0)

print(f"Algorithms: {list(algorithms.keys())}")

In [ ]:
# Train and evaluate
results = []

for name, model in algorithms.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_processed, y, cv=cv, scoring='accuracy')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    results.append({
        'Algorithm': name,
        'Test Accuracy': round(accuracy, 4),
        'CV Mean': round(cv_mean, 4),
        'CV Std': round(cv_std, 4)
    })
    
    print(f"{name}: Test={accuracy:.4f}, CV={cv_mean:.4f} (+/-{cv_std:.4f})")

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
results_df = results_df.reset_index(drop=True)
results_df.index = results_df.index + 1
results_df

## Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(results_df['Algorithm'], results_df['Test Accuracy'], color='steelblue')
ax.set_xlabel('Accuracy')
ax.set_title('Model Comparison - Test Accuracy (Raw Data)')
ax.set_xlim(0.5, 1.0)

for bar, acc in zip(bars, results_df['Test Accuracy']):
    ax.text(acc + 0.01, bar.get_y() + bar.get_height()/2, f'{acc:.4f}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 50)
print("SUMMARY - Compare with Platform Results")
print("=" * 50)
print(f"Dataset: titanic.csv ({df.shape[0]} rows)")
print(f"Raw data - no cleaning")
print(f"Split: test_size=0.2, random_state=42")
print(f"CV: 3-fold StratifiedKFold")
print("\nBest:", results_df.iloc[0]['Algorithm'])
print("Accuracy:", results_df.iloc[0]['Test Accuracy'])